In [3]:
import openai

print(openai.__version__)

2.8.1


In [4]:

import os
import sys

os.chdir("..")
sys.path.append("src/")

In [10]:
from openai import OpenAI

model = 'gpt-5.1'
client = OpenAI()

user_prompt = 'Hi, my name is Maliheh'
response = client.responses.create(
    model=model,
    input=user_prompt
)

print(response.output_text)

Nice to meet you, Maliheh. How can I help you today?


In [11]:
from pydantic import BaseModel

class EventExtractInfo(BaseModel):
    name: str
    date: str
    participants: list[str]


response = client.responses.parse(
    model = model,
    input = [
        {'role': 'system', 'content': 'Extract the event information'},
        {'role': 'user', 'content': 'Alice and Bob are going to a science fair on Friday.'}
    ],
    text_format = EventExtractInfo
)

print(response.output_parsed)

name='science fair' date='Friday' participants=['Alice', 'Bob']


In [12]:
model = "gpt-4.1-mini"

response = client.responses.create(
    model = model,
    input = [
        {
            'role': 'user',
            'content': [
                {'type': 'input_text', 'text': "what's in this image?"},
                {'type': 'input_image', 'image_url': 'https://www.eatingwell.com/thmb/8MXOea6bVolkuLNQ1HhNt1tryIE=/750x0/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/article_7866255_foods-you-should-eat-every-week-to-lose-weight_-04-d58e9c481bce4a29b47295baade4072d.jpg'}
            ]
        }
    ]
)

print(response.output_text)

The image shows a variety of healthy foods arranged on a light blue surface. The items visible include:

- A filet of salmon on a white plate
- Three eggs
- Two apples
- Two bars of dark chocolate
- A bowl of pistachio nuts
- An avocado, one half showing the seed
- Broccoli florets
- Purple cabbage slice
- Baby bok choy
- A bowl of sauerkraut
- A bowl of kimchi
- Two bowls of grains or seeds (one appears to be brown rice)
- A small bowl of chia seeds

All these items are commonly associated with nutritious and balanced diets.


In [13]:
import base64

model = "gpt-4.1"

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


image_path = '/Users/malihehnorouzi/Desktop/foods-you-should-eat-every-week-to-lose-weight.webp'
base64_image = encode_image(image_path)

response = client.responses.create(
    model=model,
    input=[
        {
            "role": "user",
            "content": [
                { "type": "input_text", "text": "what's in this image?" },
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }
    ],
)

print(response.output_text)

This image shows a variety of healthy foods arranged on a surface. The items include:

- Red cabbage
- Brown rice (in a bowl)
- Bok choy (Chinese cabbage)
- Salmon fillet (on a white plate)
- Broccolini (baby broccoli)
- Avocado (whole and halved)
- Eggs
- Sauerkraut (in a bowl)
- Apples
- Kimchi (in a bowl)
- Dark chocolate squares
- Pistachio nuts (on a plate)
- Chia seeds (in a small bowl)
- Farro or another type of whole grain (in a bowl)

These foods are rich in nutrients, fiber, healthy fats, and proteins, and are often associated with balanced diets and gut health.


In [15]:
# lets you convert files (like images) into Base64 text
import base64

def encode_image(image_path):
    # Step 1: Open the image as bytes (binary data)
    # rb: read binary (images are not text, they're raw bytes)
    image_file = open(image_path, "rb")

    # Step 2: Read all the bytes from the image
    image_bytes = image_file.read()

    # Step 3: Convert those bytes into Base64 bytes
    base64_bytes = base64.b64encode(image_bytes)

    # Step 4: Convert Base64 bytes into a normal string
    base64_string = base64_bytes.decode("utf-8")

    # Step 5: Return the final Base64 string
    return base64_string


In [16]:
import base64

raw_data = b"\x89PNG\r\n\x1a\n"  # This is how real image bytes often look
print("RAW BYTES:", raw_data)

# Step 1: Base64 encode the raw bytes
base64_bytes = base64.b64encode(raw_data)
print("BASE64 (bytes):", base64_bytes)

# Step 2: Convert Base64 bytes → string
base64_string = base64_bytes.decode("utf-8")
print("BASE64 (string):", base64_string)


RAW BYTES: b'\x89PNG\r\n\x1a\n'
BASE64 (bytes): b'iVBORw0KGgo='
BASE64 (string): iVBORw0KGgo=


In [8]:
from services.image_processing import encode_image
from services.chat_gpt.gpt import ChatGPT
from pydantic import BaseModel

class Ingredient(BaseModel):
    ingredient_name: str
    portiont: str

class Ingredients(BaseModel):
    name: str
    ingredients: list[Ingredient]
    
model = 'gpt-4.1'
system_prompt = 'Extract all the ingredients from image'
user_prompt = "what's in this image?"

image_path = '/Users/malihehnorouzi/Desktop/foods-you-should-eat-every-week-to-lose-weight.webp'
base64_image = encode_image(image_path)

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_base64_image(
    model=model, 
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    base64_image=base64_image,
    response_format = Ingredients
)

print(response)


name='Ingredients in the image' ingredients=[Ingredient(ingredient_name='Brown rice', portiont='1 small bowl'), Ingredient(ingredient_name='Red cabbage', portiont='1 wedge'), Ingredient(ingredient_name='Bok choy', portiont='3 bunches'), Ingredient(ingredient_name='Salmon', portiont='1 fillet'), Ingredient(ingredient_name='Farro (whole grain)', portiont='1 small bowl'), Ingredient(ingredient_name='Broccolini', portiont='2 stalks'), Ingredient(ingredient_name='Avocado', portiont='1.5 avocados'), Ingredient(ingredient_name='Eggs', portiont='3 eggs'), Ingredient(ingredient_name='Sauerkraut', portiont='1 bowl'), Ingredient(ingredient_name='Chia seeds', portiont='small bowl'), Ingredient(ingredient_name='Pistachios', portiont='1 plate'), Ingredient(ingredient_name='Dark chocolate', portiont='4 bars'), Ingredient(ingredient_name='Kimchi', portiont='1 bowl'), Ingredient(ingredient_name='Apple', portiont='2 whole')]


In [5]:
from services.chat_gpt.gpt import ChatGPT
from pydantic import BaseModel

class Ingredient(BaseModel):
    ingredient_name: str
    portiont: str

class Ingredients(BaseModel):
    name: str
    ingredients: list[Ingredient]

model = 'gpt-4.1'
system_prompt = 'Extract all the ingredients from image'
user_prompt = "what's in this image?"

image_url = 'https://www.eatingwell.com/thmb/8MXOea6bVolkuLNQ1HhNt1tryIE=/750x0/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/article_7866255_foods-you-should-eat-every-week-to-lose-weight_-04-d58e9c481bce4a29b47295baade4072d.jpg'

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_image_url(
    model=model, 
    user_prompt=user_prompt,
    system_prompt=system_prompt,
    image_url=image_url,
    response_format = Ingredients
)

print(response)

name='Ingredients in the Image' ingredients=[Ingredient(ingredient_name='Brown rice', portiont='1 small bowl'), Ingredient(ingredient_name='Red cabbage', portiont='1/2 head'), Ingredient(ingredient_name='Baby bok choy', portiont='3 pieces'), Ingredient(ingredient_name='Salmon fillet', portiont='1 piece'), Ingredient(ingredient_name='Broccolini', portiont='several stalks'), Ingredient(ingredient_name='Avocado', portiont='1 whole (halved)'), Ingredient(ingredient_name='Eggs', portiont='3 whole'), Ingredient(ingredient_name='Sauerkraut', portiont='1 small bowl'), Ingredient(ingredient_name='Kimchi', portiont='1 small bowl'), Ingredient(ingredient_name='Apples', portiont='2 whole'), Ingredient(ingredient_name='Pistachios', portiont='1 small plate'), Ingredient(ingredient_name='Dark chocolate', portiont='several squares'), Ingredient(ingredient_name='Chia seeds', portiont='1 small bowl'), Ingredient(ingredient_name='Farro or other whole grain', portiont='1 small bowl')]
